In [1]:
import torch
from torch import nn
import torch_geometric
from torch_geometric.nn import SAGEConv, GraphConv, GraphSAGE
import torch
# from torch_geometric.data import NeighborSampler
from torch_geometric.loader import NeighborSampler
import networkx as nx 
from torch.utils.data import Dataset, DataLoader
import pickle
import pdb 
import numpy as np
from torch_geometric.data import Data
import networkx as nx
import numpy as np
import pandas as pd 
import pickle
import pdb
from torch_geometric.loader import DataLoader
import torch.optim as optim
import torch.nn.functional as F
from torch_geometric.utils import to_dense_adj, subgraph, k_hop_subgraph
import matplotlib.pyplot as plt
import sys
import os
import pickle
sys.path.append(os.path.join(os.path.dirname(sys.path[0]),'tools'))
sys.path.append(os.path.join(os.path.dirname(sys.path[0]),'sim'))
from sklearn.neighbors import kneighbors_graph
from torch.optim.lr_scheduler import ReduceLROnPlateau
from ClassGraphDat import GraphDataset
torch.manual_seed(42)

In [2]:
class GraphEncoderWithResidual(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(GraphEncoderWithResidual, self).__init__()
        # self.conv1 = SAGEConv(in_channels, hidden_channels*4, flow='target_to_source', root_weight=False)
        # self.conv2 = SAGEConv(hidden_channels*4, hidden_channels*2, flow='target_to_source', root_weight=False)
        # self.conv3 = SAGEConv(hidden_channels*2, hidden_channels, flow='target_to_source', root_weight=False)
        self.conv1 = SAGEConv(in_channels, hidden_channels*4, flow='source_to_target', root_weight=False)
        self.conv2 = SAGEConv(hidden_channels*4, hidden_channels*2, flow='source_to_target', root_weight=False)
        self.conv3 = SAGEConv(hidden_channels*2, hidden_channels, flow='source_to_target', root_weight=False)
        self.lin1 = nn.Linear(hidden_channels, int(hidden_channels/2))
        self.lin2 = nn.Linear(int(hidden_channels/2), out_channels)
        # Linear transformation to match dimensions for residual connection
        self.shortcut = nn.Linear(in_channels, out_channels)
        # self.sm = nn.Softmax(dim=1)

    def forward(self, x, edge_index):
        identity = x
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = self.conv3(x, edge_index)
        x = F.dropout(F.relu(self.lin1(x)), p=0.2)
        x = self.lin2(x)
        # Applying shortcut and adding it to the output of conv3
        identity = self.shortcut(identity)
        x += identity  # Element-wise addition
        # x = self.sm(x)
        return x

In [3]:
# Assuming a loss function appropriate for node feature reconstruction, e.g., MSE for continuous features
def loss_function(reconstructed_x, original_x):
    # weights = torch.tensor([  2204787.0/14022   ,2204787.0/35079,  2204787.0/780448 ,2204787.0/1375238], dtype=torch.float)
    total = 1.0#2204689.0
    # arr = [1410221,  794468] 
    arr = [1.0, 1.0]
    weights = torch.tensor([total/a for a in arr], dtype = torch.float)
    
    weights = weights/torch.sum(weights)
    # return F.cross_entropy(reconstructed_x, original_x, weight=weights, label_smoothing=0.2)
    # return F.cross_entropy(reconstructed_x, original_x, label_smoothing=0.4)
    return F.cross_entropy(reconstructed_x, original_x, weight=weights)
    # return F.cross_entropy(reconstructed_x, original_x)#/scale#, label_smoothing=0.4)   

def cosine_loss(adj, emb):
    norms = torch.norm(emb, p=2, dim=1, keepdim=True)
    normalized_embeddings = emb / norms.clamp(min=1e-4)
    cosine_similarity_matrix = torch.mm(normalized_embeddings, normalized_embeddings.t())
    target_adjacency = torch.zeros_like(cosine_similarity_matrix)
    target_adjacency[adj[0], adj[1]] = 1
    # Directly using logits; no need to apply sigmoid.
    return F.binary_cross_entropy_with_logits(cosine_similarity_matrix, target_adjacency)

# from memory_profiler import profile

# @profile
def cosine_loss_batchwise(adj, emb, eDict):
    # Normalize the embeddings along the last dimension
    norms = torch.norm(emb, p=2, dim=-1, keepdim=True)
    normalized_embeddings = emb / norms.clamp(min=1e-4)
    # pdb.set_trace()
    # Calculate cosine similarity in batches to save memory
    batch_size = 10  # Adjust batch size based on your GPU/CPU memory
    losses = []
    num_nodes = emb.size(0)
    
    
    for i in range(0, num_nodes, batch_size):
        endnum = min(i + batch_size, num_nodes)
        batch_emb = normalized_embeddings[i:endnum]
        batch_cosine_sim = torch.matmul(batch_emb, normalized_embeddings.transpose(0, 1))
        
        # Creating a target matrix for the current batch
        batch_adj = torch.zeros_like(batch_cosine_sim)
        
        # Filling the target adjacency matrix only for relevant entries
        # We iterate over all edges and check if they fall into the current batch
        currentnodeIDs = list(range(i,endnum))
        # print(currentnodeIDs)
        for nodeIDnow in currentnodeIDs:
            # print(nodeIDnow)
            # print(eDict)
            for target in eDict[nodeIDnow]:
                batch_adj[nodeIDnow-i, target] = 1
        # for edge in range(adj[0].size(0)):
            # if adj[0][edge] >= i and adj[0][edge] < end:
            #     src_index = adj[0][edge] - i  # Adjust index for the current batch
            #     tgt_index = adj[1][edge]
            #     batch_adj[src_index, tgt_index] = 1
            #     batch_adj[src_index, tgt_index] = 1  # Assuming undirected graph for symmetry

        # Compute loss for this batch using binary cross-entropy with logits
        batch_loss = F.binary_cross_entropy_with_logits(batch_cosine_sim, batch_adj)
        losses.append(batch_loss)

    # Average the losses across batches
    total_loss = torch.stack(losses).mean()

    return total_loss

In [4]:
def batch(n, e):

    return nodesubset, edgesubset
def train(model, optimizer, device, nodes, edges, eDict):
    model.train()
    total_loss = 0
    embeddings = []
    origs = []
    # lossfunc = nn.CrossEntropyLoss()
    # print(data_loader)
    # printlosses = []
    # for dat in data_loader:
    # pdb.set_trace()
    # print(np.shape(nodes), np.shape(edges), nodes, edges)
    # n = dat[0] #torch.tensor(dat[1], dtype = torch.float) #nodes.squeeze_(0)
    # e = dat[1] #torch.tensor(dat[2], dtype = torch.long) #edges.squeeze_(0)
    # classes = dat[0] #torch.tensor(dat[2], dtype = torch.long)
    # class_now = get_class(meta, n)
    # print(np.shape(nodes), np.shape(edges), np.shape(class_now))

    optimizer.zero_grad()
    # adjs = [adj.to(device) for adj in adjs]
    # pdb.set_trace()
    # e = adjs[0][0]
    # optimizer.zero_grad()
    # out = model(data.x[n_id], adjs[0][0])
    # loss = F.cross_entropy(out, data.y[n_id[:batch_size]])
    # n = nt[n_id]
    counter = 0
    for n, e in batch(nodes, edges, eDict):
        counter += 1
        try:
            if n.dim() == 1:
                n = n.unsqueeze(1)

            embed = model(n, e)

        except Exception as e:
            pdb.set_trace()
        # print(eDict)
        # print(len(data[0][0]))
        # pdb.set_trace()
        # print(np.shape(class_now), class_now)
        # if np.mean(class_now.tolist()) < 0.01:
        #     pdb.set_trace()
        loss = cosine_loss_batchwise(e, embed, eDict.copy()) #cosine_loss(data[1][0].to(device), embed)
        # loss = focal_loss(embed, data[2][0].to(device)) #cosine_loss(data[1][0].to(device), embed)
        loss.backward()
        optimizer.step()
        total_loss += loss.detach().item()
        # printlosses.append(loss.detach().item())
        # if len(printlosses)%10000 == 0:
        #     printlosses = printlosses[-10000:]
        #     print(np.sum(printlosses)/10000.0)
        embeddings.append(embed.detach())
    # print(total_loss)/10.0

    return total_loss, embeddings, origs

In [5]:
folder_graph = './'
device = "cpu"
# fname = 'small_graphs.pth'
# files = os.listdir(folder_graph)
# files = [file for file in files if file.endswith('.pickle')]
files = ['alledges', 'allsucc', 'alltime', 'edgesList', 'nodeIDs']
hC = 10
inC = 40
outchannels = 3

In [6]:

# fil2 = open('MULTIPLEalltime.pickle', 'rb')
# times = pickle.load(fil2)
# fil2.close()

# fil2 = open('MULTIPLEallsucc.pickle', 'rb')
# successes = pickle.load(fil2)
# fil2.close()

fil2 = open('edgeList.pickle', 'rb')
edges = pickle.load(fil2)
fil2.close()

fil2 = open('nodeIDs.pickle', 'rb')
nodes = pickle.load(fil2)
fil2.close()

# meanTimes = dict()
# for n in times:
#     meanTimes[n] = np.mean(times[n])

In [7]:
TIME_LIMIT =  400

# def round_function(x, d):
#     new = []
#     for r in x:
#         new.append(np.round(r,decimals=d))
#     # pdb.set_trace()
#     return tuple(new)

def get_class(a):
    if a > TIME_LIMIT:
        retVal = torch.tensor(0, dtype=torch.long)
        return retVal
    else:
        retVal = torch.tensor(1, dtype=torch.long)
        return retVal
    # arr = []
    # if a > TIME_LIMIT:
    #     arr.append(0)
    # else:
    #     arr.append(1)

    # try:
    #     retVal = torch.tensor(arr, dtype=torch.long)
    #     return retVal
    # except:
    #     pdb.set_trace()
    # edges

In [8]:
# timeClasses = dict()
# for n in meanTimes:
#     timeClasses[n] = get_class(meanTimes[n])

# times_ = []
# for t in timeClasses.values():
#     times_.append(t)



def sample_neighbors(adj_list, nodes, batch_size, num_neighbors):
    # Randomly select batch_size nodes
    all_nodes = list(range(0,len(nodes)))
    batch_nodes = np.random.choice(all_nodes, batch_size, replace=False)
    
    sampled_subgraph = set()
    sampled_edges = []
    # For each node in the batch, sample num_neighbors from its neighbors
    for node in batch_nodes:
        neighbors = adj_list[node]
        if len(neighbors) > num_neighbors:
            # Sample num_neighbors if there are enough neighbors
            sampled_neighbors = np.random.choice(neighbors, num_neighbors, replace=False)
        else:
            # Otherwise take all neighbors
            sampled_neighbors = neighbors

        sampled_subgraph.update(sampled_neighbors)
        sampled_subgraph.add(node)  # Also add the node itself to the subgraph

        for neighbor in sampled_neighbors:
                sampled_edges.append((node, neighbor))
    
    sampled_nodes_tensor = torch.tensor(list(sampled_subgraph), dtype=torch.long)
    sampled_edges_tensor = torch.tensor(sampled_edges, dtype=torch.long)
    return sampled_nodes_tensor, sampled_edges_tensor




In [9]:

nodesList = []
for n in nodes:
    nodesList.append(n)

from collections import defaultdict
edgeDict = defaultdict(list)
for e in edges:
    # if e[0] not in edgeDict:
        # edgeDict[e[0]] = e[1]
        # pdb.set_trace()
    edgeDict[e[0]].append(e[1])


# pprint(edgeDict)


In [10]:


# dataset = GraphDataset(folder_graph, files)
# dataloader = DataLoader(dataset, batch_size=1, shuffle=True, num_workers=8)
# dataloader = NeighborSampler(torch.tensor(edges, dtype=torch.long).t().to(device), sizes=[10, 10, 10], batch_size=1, shuffle=True, num_workers=12)

nodeTensor = torch.tensor(nodesList, dtype=torch.float).to(device)
edgeTensor = torch.tensor(edges, dtype=torch.long).t().to(device)


actuals = []
final_loss = []
embeddings = []
dataloader = custom_loader(nodeTensor, edgeDict)
# dataloader = [torch.tensor(nodesList, dtype=torch.float).to(device), torch.tensor(edges, dtype=torch.long).t().to(device)]
model = GraphEncoderWithResidual(in_channels=inC, hidden_channels=hC, out_channels=outchannels).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.1)
# scheduler = ReduceLROnPlateau(optimizer, 'min', factor=0.1, patience=3, verbose=True)
losses = []
embeds = []
acts = []


In [11]:
# dataloader[2]
# np.shape(dataloader[2].t())
# ! python3 -m pip install torch-sparse


In [12]:
# print(edgeDict[0])
# embeddings = []
embeds = []
for epoch in range(1, 30):  # Number of epochs
    # sampled_nodes, sampled_edges = sample_neighbors(edges, nodesList, batch_size=10, num_neighbors=10)
    # sampled_features = nodeTensor[sampled_nodes]
    loss, embed, orig = train(model, optimizer, device, nodeTensor, edgeTensor, edgeDict)
    embeds.append(embed)
    # acts.append(orig)
    print(f'Epoch {epoch}, Loss: {loss:.4f}')
    # losses.append(loss)
    # scheduler.step(loss)
# epoch_loss = loss / len(dataloader)
# embeddings.append(embeds)
# final_loss.append(losses)
# actuals.append(acts)
model.eval()
# Assuming encoder and decoder are your model's components
encoder_state_dict = model.state_dict()
# decoder_state_dict = model.decoder.state_dict()
# Save the state dictionaries
torch.save(encoder_state_dict, './Embed_encoder_state_dict'+str(outchannels)+'.pth')
# torch.save(decoder_state_dict, './CDC/decoder_state_dict'+str(outchannels)+'.pth')
        # print(np.shape(embed), np.shape(data[2][0]), np.shape(data[0][0]), np.shape(data[1][0]))


ERROR: Could not find file /var/folders/ng/kxgs87g91gbb9s1st4h8n99m0000gq/T/ipykernel_91400/86939875.py


KeyboardInterrupt: 

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=0.01)

for epoch in range(1, 30):  # Number of epochs
    loss, embed, orig = train(model, optimizer, device, nodeTensor, edgeTensor)
    # embeds.append(embed)
    # acts.append(orig)
    print(f'Epoch {epoch}, Loss: {loss:.4f}')
    # losses.append(loss)
model.eval()
encoder_state_dict = model.state_dict()
torch.save(encoder_state_dict, './Embed_encoder_state_dict'+str(outchannels)+'.pth')


> /var/folders/ng/kxgs87g91gbb9s1st4h8n99m0000gq/T/ipykernel_90619/4234558143.py(30)cosine_loss_batchwise()
     28     pdb.set_trace()
     29     # Calculate cosine similarity in batches to save memory
---> 30     batch_size = 50  # Adjust batch size based on your GPU/CPU memory
     31     losses = []
     32     num_nodes = emb.size(0)



In [ ]:
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(1, 30):  # Number of epochs
    loss, embed, orig = train(model, optimizer, device, nodeTensor, edgeTensor)
    # embeds.append(embed)
    # acts.append(orig)
    print(f'Epoch {epoch}, Loss: {loss:.4f}')
    # losses.append(loss)
model.eval()
encoder_state_dict = model.state_dict()
torch.save(encoder_state_dict, './Embed_encoder_state_dict'+str(outchannels)+'.pth')

> /var/folders/ng/kxgs87g91gbb9s1st4h8n99m0000gq/T/ipykernel_90619/4234558143.py(30)cosine_loss_batchwise()
     28     pdb.set_trace()
     29     # Calculate cosine similarity in batches to save memory
---> 30     batch_size = 50  # Adjust batch size based on your GPU/CPU memory
     31     losses = []
     32     num_nodes = emb.size(0)

--KeyboardInterrupt--

KeyboardInterrupt: Interrupted by user


In [ ]:
optimizer = optim.Adam(model.parameters(), lr=0.0001)

for epoch in range(1, 30):  # Number of epochs
    loss, embed, orig = train(model, optimizer, device, nodeTensor, edgeTensor)
    # embeds.append(embed)
    # acts.append(orig)
    print(f'Epoch {epoch}, Loss: {loss:.4f}')
    # losses.append(loss)
model.eval()
encoder_state_dict = model.state_dict()
torch.save(encoder_state_dict, './Embed_encoder_state_dict'+str(outchannels)+'.pth')

In [ ]:
# # GAE = GraphEncoderWithResidual(inC, hC, outchannels)
# # import pickle
# import numpy as np
# import torch


# from collections import defaultdict 

# encoder_dict = torch.load('./Time_encoder_state_dict2.pth')


In [ ]:

# GAE = GraphEncoderWithResidual(inC, hC, outchannels)
# GAE.load_state_dict(encoder_dict)
# emb = []
# # arrtosave = []

In [ ]:
# print(np.shape(torch.tensor([nodesList[0]], dtype=torch.float)), np.shape(torch.tensor([[],[]], dtype=torch.long)))
# # GAE(torch.tensor([nodesList[0]], dtype=torch.float), torch.tensor([[],[]], dtype=torch.long))


In [ ]:

# # for i in arr1: 
# GAE.eval()
# emb = []
# for ii in nodesList: 
#     # break
# # GAE(torch.tensor([ast.literal_eval(i)], dtype=torch.float), torch.empty((2,0), dtype=torch.int64).detach().tolist()[0])
#     try:
#         # pdb.set_trace()
#         emb.append(GAE(torch.tensor([ii], dtype=torch.float), torch.tensor([[],[]], dtype=torch.long)).detach().tolist()[0])
#         # arrtosave.append(i) 
#     except Exception as e:
#         print(e)
#         print(np.shape(ii))

In [ ]:
# times_
# times_check = []
# for t in times_:
#     times_check.append(t.tolist())

In [ ]:
# import numpy as np
# from scipy.special import expit, logit
# emb
# newembs = []
# for e in emb:
#     newembs.append(expit(e))

# newembs
# emb

In [ ]:
# print(np.shape(times_check), np.shape(emb))

# y_true_indices = times_check#np.argmax(times_check, axis=1)
# y_pred_indices = np.argmax(emb, axis=1)
# # print(np.argmax(emb, axis=1)[0:10])
# # print(np.argmax(arrclass, axis=1)[0:10])


In [ ]:
# from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
# import seaborn as sns
# cm = confusion_matrix(y_true_indices, y_pred_indices)
# # ConfusionMatrixDisplay(cm).plot()
# # CLIST =  ["Fast/Failure", "Slow/Failure", "Fast/Success", "Slow/Success"]

# CLIST =  ["SLOW", "FAST"]
# # Plotting the confusion matrix
# plt.figure(figsize=(8, 6))
# sns.heatmap(cm, annot=True, fmt="d", cmap='Blues', xticklabels=CLIST, yticklabels=CLIST)
# plt.xlabel('Predicted Labels')
# plt.ylabel('True Labels')
# plt.title('Confusion Matrix')
# plt.show()
# 

In [ ]:
# from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, accuracy_score

In [ ]:
# print('roc auc score: ', roc_auc_score(y_true_indices, y_pred_indices))
# print('precision score: ', precision_score(y_true_indices, y_pred_indices))
# print('recall score: ', recall_score(y_true_indices, y_pred_indices))
# print('f1 score: ', f1_score(y_true_indices, y_pred_indices))
# print('accuracy score: ', accuracy_score(y_true_indices, y_pred_indices))


10:

roc auc score:  0.7847262434011781
precision score:  0.7394655295706757
recall score:  0.7593559617058312
f1 score:  0.7492787655149278
accuracy score:  0.7890321560834613

5: 

roc auc score:  0.7807121435124047
precision score:  0.7352067868504772
recall score:  0.7542428198433421
f1 score:  0.7446031575555795
accuracy score:  0.7852045885647186

In [ ]:


# print(np.sum([1 for y in y_true_indices if y == 1]))
# total = np.sum(cm, axis=1)
# print(total, np.sum(total))
# print(cm)
# 0: 14022, 1: 35079, 2: 780448 1375238

In [ ]:
# import torch
# import statistics
# arr = []
# for a in meta:
#     arr.append(statistics.mean(meta[a]))

# times_ = []
# for t in timeClasses.values():
#     times_.append(t)
# # print(n)
# import matplotlib.pyplot as plt
# x=plt.hist(times_)
# plt.xlim([0,200])


# import shutil
# import os
# import numpy as np
# import pandas as pd
# from ast import literal_eval
# from collections import defaultdict
# def other():
#     folder = '../../graphsage_results/CDC/multiple_agent_env_results/'
#     files = os.listdir(folder)
#     files = [file for file in files if file.endswith('.csv') and file.startswith('1')]
#     files = np.sort(files)
#     # data_files = []
#     metadata_file = folder + 'metadata.csv'
#     # folder_graph = './graphs/RS/fast_multiple_graphs/'
#     # new_metadata_file = folder_graph + 'metadata.csv'
#     # graph_metaFile = 'graphMetadata.csv'
#     # meta_arr = []
#     metadata = pd.read_csv(metadata_file) 
#     metadata.site_qualities=metadata.site_qualities.apply(literal_eval)
#     metadata.site_positions=metadata.site_positions.apply(literal_eval)
#     metadata.site_positions=metadata.site_positions.apply(lambda x: tuple([tuple(a) for a in x]))
#     metadata.site_qualities=metadata.site_qualities.apply(lambda x: tuple(x))
#     df = metadata.groupby(by=['site_qualities', 'site_positions', 'num_agents'], as_index=False).agg(lambda x: x.tolist())
#     Gtime = defaultdict(list)
#     Gsucc = defaultdict(list)
#     Gedge = defaultdict(defaultdict)
    
#     for some_id, entry in enumerate(df.iterrows()):
#         # copy files
#         for file in entry[1].iloc[3]:
#             shutil.copyfile(folder+file, '../../oneGraphFiles/'+file)
#         break

# other()